# Tutorial 3: Multi-Camera 3D Triangulation

**Pipeline Stage:** Fusing matched 2D poses into 3D skeletons

---

## Overview

This tutorial covers the **third and final stage** of the multi-camera human tracking
pipeline: taking the matched 2D poses and **triangulating** them into 3D skeleton
coordinates.

### Why Triangulation Comes LAST

Triangulation requires two inputs:
1. **2D keypoints per camera** (from Tutorial 1: YOLO Pose)
2. **Cross-camera identity correspondences** (from Tutorial 2: ReID)

Without knowing which detection in Camera 1 is the same person as which detection in
Camera 3 (the ReID step), triangulation would try to combine keypoints from *different
people* across cameras — producing nonsensical 3D skeletons.

```
The problem without ReID:
  CAM1: Person A at (100, 200)     CAM3: Person X at (300, 150)
  CAM1: Person B at (500, 300)     CAM3: Person Y at (450, 250)
                                   CAM3: Person Z at (100, 400)

  Which pairs do we triangulate?
  A+X? A+Y? A+Z? B+X? B+Y? B+Z?  ← ReID tells us!
```

### What You Will Learn

1. **Installation** — `sleap-anipose`, `h5py`, `matplotlib`, and dependencies
2. **Understanding the Input Data** — H5 pose files from Tutorial 1, identity map from Tutorial 2
3. **Inspecting H5 Pose Files** — Structure, shapes, track counts, node names
4. **Camera Calibration** — What a calibration TOML file contains
5. **Running Triangulation** — Using `sleap-anipose` to produce 3D points
6. **Inspecting 3D Results** — Loading and understanding the output H5 file
7. **3D Visualization** — Plotting skeletons in 3D with matplotlib
8. **Scene Normalization** — Orienting the coordinate system so Z = up, floor = Z=0

### Pipeline Context

```
┌─────────────────────┐     ┌──────────────────────┐     ┌─────────────────────┐
│  Tutorial 1          │ ──► │  Tutorial 2           │ ──► │  Tutorial 3 (HERE)   │
│  YOLO Pose (2D)     │     │  Person ReID          │     │  3D Triangulation    │
│  per-camera          │     │  cross-camera match   │     │  multi-camera fusion │
└─────────────────────┘     └──────────────────────┘     └─────────────────────┘
       outputs:                    outputs:                    outputs:
    2D keypoints per cam      identity map (who=who)      3D skeleton coords
```

### Prerequisites

- Completed Tutorial 1 (2D pose results in `.slp` / `.analysis.h5` format per camera)
- Completed Tutorial 2 (cross-camera identity correspondences)
- Camera calibration file (`.toml`) for your multi-camera rig
- 6 synchronized camera views (our setup: CAM1-CAM6)

---

## Part 1: Installation

| Package | Purpose | Install |
|---|---|---|
| `sleap-anipose` | Multi-camera triangulation engine | `pip install sleap-anipose` |
| `h5py` | Read/write HDF5 pose data files | `pip install h5py` |
| `matplotlib` | 3D visualization | `pip install matplotlib` |
| `seaborn` | Color palettes for plots | `pip install seaborn` |
| `numpy` | Array operations | `pip install numpy` |
| `pandas` | Tabular data inspection | `pip install pandas` |

### About sleap-anipose

`sleap-anipose` bridges SLEAP pose estimation with the Anipose triangulation library.
It handles:
- Loading 2D poses from multiple camera views
- Applying camera calibration parameters
- Running Direct Linear Transformation (DLT) triangulation
- Optional spatiotemporal smoothing and limb-length constraints

### How Triangulation Works (Conceptually)

```
Camera 1 sees nose at pixel (320, 240)
Camera 2 sees nose at pixel (510, 195)
Camera 3 sees nose at pixel (280, 310)
         │
         ▼  (using calibration: intrinsics + extrinsics)
Cast a ray from each camera through its 2D detection
         │
         ▼
Find the 3D point that best satisfies all rays
         │
         ▼
nose_3d = (1.42, 0.85, 1.73) meters
```

In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================
# Uncomment and run if not yet installed:

# !pip install sleap-anipose h5py matplotlib seaborn numpy pandas

In [ ]:
# ============================================================
# STEP 2: Verify installation
# ============================================================
import h5py
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print(f"h5py:       {h5py.__version__}")
print(f"numpy:      {np.__version__}")
print(f"pandas:     {pd.__version__}")
print(f"matplotlib: {matplotlib.__version__}")

try:
    import sleap_anipose
    print(f"sleap-anipose: available")
except ImportError:
    print("WARNING: sleap-anipose not installed. Install with: pip install sleap-anipose")

---

## Part 2: Understanding the Inputs

Triangulation needs two things from the earlier tutorials:

### From Tutorial 1: Per-Camera 2D Pose H5 Files

Each `.analysis.h5` file contains:

```
├── tracks          shape: (n_instances, 2, n_nodes, n_frames)
│                          - 2 = (x, y) coordinates
│                          - n_nodes = 17 (COCO keypoints)
│                          - n_frames = number of video frames
├── track_names     list of track identifiers
├── node_names      list of keypoint names
├── edge_inds       skeleton connections
└── track_occupancy which tracks are present in which frames
```

### From Tutorial 2: Cross-Camera Identity Map

The ReID step tells us: for each frame, which instance in Camera A is the same person
as which instance in Camera B. This lets `sleap-anipose` (or our custom triangulation)
combine the correct 2D points.

### File Organization

```
triangulation_7_18_25/
├── cam1/  *.analysis.h5      ← Tutorial 1 output
├── cam2/  *.analysis.h5
├── cam3/  *.analysis.h5
├── cam4/  *.analysis.h5
├── cam5/  *.analysis.h5
├── cam6/  *.analysis.h5
├── calibration.toml           ← Camera calibration
└── identity_map.json          ← Tutorial 2 output (ReID)
```

In [ ]:
# ============================================================
# STEP 3: Inspect the per-camera H5 pose files
# ============================================================
import h5py
import os
import numpy as np
import pandas as pd

# Define paths to all camera H5 files
h5_paths = [
    "/root/vast/eric/jernigan_chiba_camp/triangulation_7_18_25/cam1/aligned_CAM1_1_28_00_to_1_30_00_pose.000_aligned_CAM1_1_28_00_to_1_30_00.analysis.h5",
    "/root/vast/eric/jernigan_chiba_camp/triangulation_7_18_25/cam2/aligned_CAM2_1_28_00_to_1_30_00_pose.000_aligned_CAM2_1_28_00_to_1_30_00.analysis.h5",
    "/root/vast/eric/jernigan_chiba_camp/triangulation_7_18_25/cam3/aligned_CAM3_1_28_00_to_1_30_00_pose.000_aligned_CAM3_1_28_00_to_1_30_00.analysis.h5",
    "/root/vast/eric/jernigan_chiba_camp/triangulation_7_18_25/cam4/aligned_CAM4_1_28_00_to_1_30_00_pose.000_aligned_CAM4_1_28_00_to_1_30_00.analysis.h5",
    "/root/vast/eric/jernigan_chiba_camp/triangulation_7_18_25/cam5/aligned_CAM5_1_28_00_to_1_30_00_pose.000_aligned_CAM5_1_28_00_to_1_30_00.analysis.h5",
    "/root/vast/eric/jernigan_chiba_camp/triangulation_7_18_25/cam6/aligned_CAM6_1_28_00_to_1_30_00_pose.000_aligned_CAM6_1_28_00_to_1_30_00.analysis.h5"
]

summary_rows = []

for path in h5_paths:
    camera = os.path.basename(os.path.dirname(path))
    
    try:
        with h5py.File(path, 'r') as f:
            print(f"\n{'='*50}")
            print(f"Camera: {camera.upper()}")
            print(f"{'='*50}")
            print(f"File size: {os.path.getsize(path) / (1024*1024):.2f} MB")
            
            # List all datasets
            print(f"\nDatasets:")
            for key in f.keys():
                shape = f[key].shape if hasattr(f[key], 'shape') else 'group'
                print(f"  {key}: {shape}")
            
            # Examine tracks
            if 'tracks' in f:
                tracks = f['tracks']
                print(f"\nTracks shape: {tracks.shape}")
                print(f"  ({tracks.shape[0]} instances, "
                      f"{tracks.shape[1]} coords, "
                      f"{tracks.shape[2]} nodes, "
                      f"{tracks.shape[3]} frames)")
                
                data = tracks[()]
                non_nan = np.count_nonzero(~np.isnan(data))
                total = np.prod(data.shape)
                print(f"  Data coverage: {non_nan/total*100:.1f}% non-NaN")
                
                summary_rows.append({
                    'Camera': camera,
                    'Instances': tracks.shape[0],
                    'Nodes': tracks.shape[2],
                    'Frames': tracks.shape[3],
                    'Coverage %': f"{non_nan/total*100:.1f}"
                })
            
            # Show node names
            if 'node_names' in f:
                names = [n.decode() if isinstance(n, bytes) else n 
                         for n in f['node_names'][()]]
                print(f"  Node names ({len(names)}): {names[:5]}...")
                
    except FileNotFoundError:
        print(f"  FILE NOT FOUND: {path}")

# Summary table
if summary_rows:
    print(f"\n\n{'='*50}")
    print("SUMMARY")
    print(f"{'='*50}")
    print(pd.DataFrame(summary_rows).to_string(index=False))

---

## Part 3: Camera Calibration

### What is Camera Calibration?

To triangulate 2D points from multiple cameras into 3D, you need to know:

1. **Intrinsic parameters** — focal length, principal point, distortion (per camera)
   - These describe the camera's internal optics
   - How a 3D point maps to a pixel location

2. **Extrinsic parameters** — rotation and translation (per camera)
   - These describe where the camera is in the world
   - Position and orientation of each camera

### Calibration TOML Format

```toml
[cam1]
  size = [1920, 1080]                              # image resolution
  matrix = [[fx, 0, cx], [0, fy, cy], [0, 0, 1]]  # intrinsic matrix
  distortions = [k1, k2, p1, p2, k3]               # lens distortion
  rotation = [r1, r2, r3]                           # Rodrigues rotation
  translation = [tx, ty, tz]                        # camera position
```

### How to Create a Calibration File

- Use a **checkerboard** or **ChArUco board** visible to all cameras simultaneously
- Tools: OpenCV `calibrateCamera()`, Anipose calibration, or Caltech toolbox
- The calibration maps each camera name to its intrinsic and extrinsic parameters

In [ ]:
# ============================================================
# STEP 4: Inspect the calibration file
# ============================================================
calib_path = "/root/vast/eric/jernigan_chiba_camp/triangulation_7_18_25/calibration.toml"

if os.path.exists(calib_path):
    with open(calib_path, 'r') as f:
        content = f.read()
    print("Calibration file contents (first 2000 chars):")
    print(content[:2000])
    if len(content) > 2000:
        print(f"\n... ({len(content)} total characters)")
else:
    print(f"Calibration file not found at: {calib_path}")
    print("You need to create a calibration file for your camera rig.")

---

## Part 4: Running Triangulation with sleap-anipose

### The Process

1. For each frame, gather the 2D keypoint positions from all cameras
2. **Use the identity map (from Tutorial 2)** to match detections across cameras
3. For each matched person, for each keypoint, cast rays from each camera
4. Find the 3D point minimizing distance to all rays (DLT)
5. Optionally apply spatiotemporal smoothing and limb-length constraints

### Key Parameters

| Parameter | Description | Typical Value |
|---|---|---|
| `p2d` | Directory containing per-camera pose H5 files | path to folder |
| `calib` | Path to calibration TOML | `calibration.toml` |
| `fname` | Output filename for 3D points | `points3d.h5` |
| `scale_smooth` | Smoothing weight (higher = smoother) | 2 |
| `scale_length` | Limb length constraint weight | 2 |
| `n_deriv_smooth` | Order of derivative for smoothing | 2 |

In [ ]:
# ============================================================
# STEP 5: Run triangulation
# ============================================================
# NOTE: This step requires the actual data files, calibration,
# and the identity correspondences from Tutorial 2.
# Uncomment to run when everything is in place.

# import sleap_anipose as slap
#
# points3d = slap.triangulate(
#     p2d='/root/vast/eric/jernigan_chiba_camp/triangulation_7_18_25',
#     calib='/root/vast/eric/jernigan_chiba_camp/triangulation_7_18_25/calibration.toml',
#     fname='points3d.h5',
#     scale_smooth=2,
#     scale_length=2,
#     scale_length_weak=2,
#     n_deriv_smooth=2,
#     verbose=True,
# )
#
# print("Triangulation complete!")
# print(f"Output saved to: {points3d}")

---

## Part 5: Inspecting the 3D Results

After triangulation, we get an H5 file with 3D skeleton coordinates.

### Output H5 Structure

```
points3d.h5
└── tracks    shape: (n_frames, n_tracks, n_joints, 3)
                     - n_tracks = matched individuals (from ReID)
                     - n_joints = 17 COCO keypoints
                     - 3 = (x, y, z) coordinates
                     - NaN for joints that couldn't be triangulated
```

In [ ]:
# ============================================================
# STEP 6: Load and inspect the 3D triangulated data
# ============================================================
import h5py
import numpy as np

file_path = "/root/vast/eric/jernigan_chiba_camp/triangulation_7_18_25/points3d_2.h5"

with h5py.File(file_path, 'r') as f:
    print("H5 File Keys:")
    for key in f.keys():
        ds = f[key]
        print(f"  {key}: shape={ds.shape}, dtype={ds.dtype}")
    
    data_key = 'tracks' if 'tracks' in f else 'points'
    data = f[data_key][()]

num_frames, num_tracks, num_joints, num_dims = data.shape

print(f"\nData shape: {data.shape}")
print(f"  Frames:     {num_frames}")
print(f"  Tracks:     {num_tracks} (matched individuals from ReID)")
print(f"  Joints:     {num_joints} (COCO keypoints)")
print(f"  Dimensions: {num_dims} (x, y, z)")

# Data quality
nan_count = np.isnan(data).sum()
total = np.prod(data.shape)
print(f"\nData quality:")
print(f"  NaN values: {nan_count:,} / {total:,} ({nan_count/total*100:.1f}%)")
print(f"  Valid data: {total - nan_count:,} ({(total-nan_count)/total*100:.1f}%)")

# Show sample keypoints for frame 0, track 0
keypoint_names = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

print(f"\nSample — Frame 0, Track 0:")
for i, name in enumerate(keypoint_names):
    x, y, z = data[0, 0, i]
    status = "valid" if not np.isnan(x) else "NaN"
    print(f"  {name:<18}: ({x:>8.2f}, {y:>8.2f}, {z:>8.2f})  [{status}]")

---

## Part 6: 3D Visualization

Let's plot the 3D skeletons using matplotlib's 3D projection.

### COCO Skeleton Connections

```
Face:  nose-eyes, eyes-ears
Arms:  shoulder-elbow-wrist
Torso: shoulder-shoulder, shoulder-hip, hip-hip
Legs:  hip-knee-ankle
```

In [ ]:
# ============================================================
# STEP 7: Plot a single 3D frame with skeleton connections
# ============================================================
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Skeleton connections
skeleton_connections = [
    (0, 1), (0, 2), (1, 3), (2, 4),           # Face
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),  # Arms
    (5, 11), (6, 12), (11, 12),                # Torso
    (11, 13), (13, 15), (12, 14), (14, 16)     # Legs
]

track_colors = ['red', 'blue', 'green', 'purple', 'orange', 'brown']


def plot_3d_frame(data, frame_idx, ax=None):
    """Plot 3D skeletons for all tracks in a given frame."""
    if ax is None:
        fig = plt.figure(figsize=(12, 10))
        ax = fig.add_subplot(111, projection='3d')
    
    frame_data = data[frame_idx]
    
    for track_idx in range(frame_data.shape[0]):
        kpts = frame_data[track_idx]  # (17, 3)
        color = track_colors[track_idx % len(track_colors)]
        
        if np.isnan(kpts).all():
            continue
        
        valid = ~np.isnan(kpts[:, 0])
        ax.scatter(kpts[valid, 0], kpts[valid, 1], kpts[valid, 2],
                   color=color, s=50, label=f"Person {track_idx}")
        
        for j1, j2 in skeleton_connections:
            if valid[j1] and valid[j2]:
                ax.plot([kpts[j1, 0], kpts[j2, 0]],
                        [kpts[j1, 1], kpts[j2, 1]],
                        [kpts[j1, 2], kpts[j2, 2]],
                        color=color, linewidth=2)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'Frame {frame_idx}')
    ax.legend()
    return ax


# Plot frame 0
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')
plot_3d_frame(data, 0, ax)
plt.tight_layout()
plt.savefig("3d_skeleton_frame.png", dpi=150)
plt.show()
print("Saved: 3d_skeleton_frame.png")

In [ ]:
# ============================================================
# STEP 8: Plot multiple frames to see motion over time
# ============================================================
sample_frames = np.linspace(0, num_frames - 1, 4, dtype=int)

fig, axes = plt.subplots(1, 4, figsize=(24, 6),
                          subplot_kw={'projection': '3d'})

for ax, fidx in zip(axes, sample_frames):
    plot_3d_frame(data, fidx, ax)

plt.suptitle('3D Skeletons Over Time', fontsize=16)
plt.tight_layout()
plt.savefig("3d_skeleton_sequence.png", dpi=150)
plt.show()
print("Saved: 3d_skeleton_sequence.png")

---

## Part 7: Scene Normalization

Raw triangulated coordinates are in an arbitrary coordinate system defined by
the camera calibration. For analysis, we want:

- **Z axis = up** (perpendicular to the floor)
- **Floor at Z = 0**
- **X-Y plane = ground plane**

### Strategy

1. **Find "up"**: Average foot-to-nose vectors across all frames/tracks
2. **Build rotation matrix**: Align this direction with the Z axis (Rodrigues' formula)
3. **Find floor level**: Use average ankle Z position after rotation
4. **Translate**: Shift so floor is at Z = 0

In [ ]:
# ============================================================
# STEP 9: Normalize the 3D scene — Z = up, floor at Z = 0
# ============================================================
import seaborn as sns
sns.set_theme(style="ticks")

# COCO keypoint indices
NOSE = 0
LEFT_ANKLE = 15
RIGHT_ANKLE = 16

normalized_data = np.copy(data)

# --- Step 1: Collect foot-to-nose vectors to determine "up" ---
up_vectors = []
foot_positions = []

for frame_idx in range(num_frames):
    for track_idx in range(num_tracks):
        pose = data[frame_idx, track_idx]
        nose = pose[NOSE]
        left_ankle = pose[LEFT_ANKLE]
        right_ankle = pose[RIGHT_ANKLE]
        
        # Average foot position
        if not np.isnan(left_ankle).any() and not np.isnan(right_ankle).any():
            feet = (left_ankle + right_ankle) / 2
            foot_positions.append(feet)
        elif not np.isnan(left_ankle).any():
            feet = left_ankle
            foot_positions.append(feet)
        elif not np.isnan(right_ankle).any():
            feet = right_ankle
            foot_positions.append(feet)
        else:
            continue
        
        if not np.isnan(nose).any():
            vec = nose - feet
            if np.linalg.norm(vec) > 0.1:
                up_vectors.append(vec / np.linalg.norm(vec))

avg_up = np.mean(up_vectors, axis=0)
avg_up = avg_up / np.linalg.norm(avg_up)
print(f"Average 'up' direction (raw): {avg_up}")

# --- Step 2: Build rotation to align 'up' with Z axis ---
target_up = np.array([0, 0, 1])
rotation_axis = np.cross(avg_up, target_up)
rotation_axis_norm = np.linalg.norm(rotation_axis)

if rotation_axis_norm < 1e-6:
    R = np.eye(3) if np.dot(avg_up, target_up) > 0 else -np.eye(3)
    angle = 0.0
else:
    rotation_axis = rotation_axis / rotation_axis_norm
    cos_angle = np.clip(np.dot(avg_up, target_up), -1, 1)
    angle = np.arccos(cos_angle)
    
    # Rodrigues' rotation formula
    K = np.array([
        [0, -rotation_axis[2], rotation_axis[1]],
        [rotation_axis[2], 0, -rotation_axis[0]],
        [-rotation_axis[1], rotation_axis[0], 0]
    ])
    R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * K @ K

print(f"Rotation angle: {np.degrees(angle):.1f} degrees")

# --- Step 3: Apply rotation ---
for frame_idx in range(num_frames):
    for track_idx in range(num_tracks):
        pose = data[frame_idx, track_idx]
        valid = ~np.isnan(pose[:, 0])
        if valid.any():
            normalized_data[frame_idx, track_idx, valid] = (R @ pose[valid].T).T

# --- Step 4: Translate so floor = Z=0 ---
all_ankle_z = []
for frame_idx in range(num_frames):
    for track_idx in range(num_tracks):
        for ankle_idx in [LEFT_ANKLE, RIGHT_ANKLE]:
            z = normalized_data[frame_idx, track_idx, ankle_idx, 2]
            if not np.isnan(z):
                all_ankle_z.append(z)

floor_z = np.percentile(all_ankle_z, 5)
normalized_data[:, :, :, 2] -= floor_z

print(f"Floor level (before shift): {floor_z:.3f}")
print(f"After normalization: Z=0 is the floor, Z>0 is up")

In [ ]:
# ============================================================
# STEP 10: Visualize the normalized 3D scene
# ============================================================
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

frame_idx = 0
frame_data = normalized_data[frame_idx]
colors = sns.color_palette("muted", n_colors=num_tracks)

for track_idx in range(num_tracks):
    kpts = frame_data[track_idx]
    valid = ~np.isnan(kpts[:, 0])
    if not valid.any():
        continue
    
    color = colors[track_idx]
    ax.scatter(kpts[valid, 0], kpts[valid, 1], kpts[valid, 2],
               color=color, s=60, label=f"Person {track_idx}")
    
    for j1, j2 in skeleton_connections:
        if valid[j1] and valid[j2]:
            ax.plot([kpts[j1, 0], kpts[j2, 0]],
                    [kpts[j1, 1], kpts[j2, 1]],
                    [kpts[j1, 2], kpts[j2, 2]],
                    color=color, linewidth=2.5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z (up)')
ax.set_title(f'Normalized 3D Scene — Frame {frame_idx}\n(Z=0 is floor, Z>0 is up)')
ax.legend()

plt.tight_layout()
plt.savefig("normalized_3d_scene.png", dpi=150)
plt.show()
print("Saved: normalized_3d_scene.png")

---

## Summary — Full Pipeline

### The Complete Pipeline (Correct Order)

```
Raw Video (6 synchronized cameras)
    │
    ▼  Tutorial 1: YOLO Pose Estimation
2D Poses per camera (.slp, .json)
  - Detects people and 17 keypoints per camera independently
  - Handles landscape and portrait cameras
    │
    ▼  Tutorial 2: Person Re-Identification
Cross-Camera Identity Map
  - Crops people/torsos from each camera
  - Extracts OSNet appearance embeddings
  - Matches same person across cameras per frame
  - MUST happen before triangulation!
    │
    ▼  Tutorial 3: 3D Triangulation (this tutorial)
3D Skeleton Data (points3d.h5)
  - Uses identity map to pair correct detections
  - Triangulates matched 2D keypoints into 3D
  - Normalizes scene (Z=up, floor=0)
    │
    ▼
Consistent 3D Tracking with Identity
  - Each person has a consistent ID across all cameras
  - Full 3D skeleton trajectory over time
  - Ready for downstream analysis (gait, behavior, etc.)
```

### Key Takeaways

- **Order matters**: Pose -> ReID -> Triangulation (not Pose -> Triangulation -> ReID)
- Triangulation quality depends on **calibration accuracy**, **ReID accuracy**, and **pose consistency**
- NaN values indicate joints that couldn't be reliably triangulated (occluded in too many views)
- Scene normalization is essential for downstream analysis

### Possible Extensions

- Temporal smoothing of 3D trajectories
- Limb-length constraints for more physically plausible skeletons
- Iterative refinement: use 3D positions to improve ReID matches
- Real-time pipeline by processing frames as they arrive